# 11. LightGBM: оценка всех моделей (Kaggle)

Собирает прогнозы обеих частей ноутбука 10, считает бейзлайны и оценивает всё по протоколу.

**Input:** датасет `vkr-transit-data` и **вывод обеих версий ноутбука 10** (Add Input → Your Work → Notebooks:
части A и B). Accelerator — None, Internet — On. Работает ~15 мин. Результат — `results_csv.zip` в Output.

In [ ]:
import os, glob, shutil, subprocess, time
REPO = "/kaggle/working/vkr-transit"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Kinggl3/vkr-transit", REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
lab = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/ridership_wide.parquet", recursive=True)]
assert lab, "Не найден ridership_wide.parquet: подключите датасет vkr-transit-data (Add Input)"
os.environ["VKR_LAB"] = lab[0]
os.environ["VKR_RESULTS"] = R = "/kaggle/working/results"
os.environ["PYTHONWARNINGS"] = "ignore"
os.makedirs(f"{R}/preds", exist_ok=True)
print("Данные:", lab[0], "| граф:", os.path.exists(os.path.join(os.path.dirname(lab[0]), "graph")),
      "| events.csv:", os.path.exists(os.path.join(lab[0], "events.csv")))
print("Подключено в /kaggle/input:")
for p in sorted(glob.glob("/kaggle/input/*/*")):
    print("  ", p)

## 1. Прогнозы частей A и B, бейзлайны

In [ ]:
prev = sorted(glob.glob("/kaggle/input/**/results/preds/*.npz", recursive=True))
print("Найдено прогнозов в подключённых выводах:", len(prev))
for p in prev:
    dst = f"{R}/preds/{os.path.basename(p)}"
    if not os.path.exists(dst):
        shutil.copy(p, dst)
# журнал запусков и важность признаков из обеих частей
logs = [p for p in glob.glob("/kaggle/input/**/results/runs.csv", recursive=True)]
import pandas as pd
if logs:
    pd.concat([pd.read_csv(p) for p in logs]).to_csv(f"{R}/runs.csv", index=False)
for p in glob.glob("/kaggle/input/**/results/fi_*.csv", recursive=True):
    shutil.copy(p, R)
!cd {REPO} && python src/baselines.py
have = sorted(os.path.basename(p) for p in glob.glob(f"{R}/preds/*.npz"))
print(have)
missing = [m for m in ["lgbm", "lgbm_all", "lgbm_line", "lgbm_dist", "lgbm_od_in", "lgbm_od_dyn", "lgbm_wactual"]
           if f"{m}__val.npz" not in have]
print("Нет прогнозов:", missing or "—")

## 2. Оценка

In [ ]:
!cd {REPO} && python src/evaluate.py --split val --ref norm_level

In [ ]:
!cd {REPO} && python src/evaluate.py --split test --ref norm_level

## 3. Горизонты, разрезы, важность признаков

In [ ]:
pd.set_option("display.width", 200, "display.max_columns", 20)
for split in ["val", "test"]:
    if not os.path.exists(f"{R}/metrics_{split}.csv"):
        continue
    m = pd.read_csv(f"{R}/metrics_{split}.csv")
    C = m[m.subset == "общий подсет"]
    hz = C[C.slice.str.startswith("горизонт")].pivot_table(index="model", columns="slice", values="wape")
    hz = hz[["горизонт h1", "горизонт h2-3", "горизонт h4-6", "горизонт h7-12", "горизонт h13-24"]]
    print(f"\n{split}: WAPE на общем подсете по горизонтам"); display(hz.round(4).sort_values("горизонт h1"))
    if "lgbm" in hz.index:
        print("изменение к lgbm без соседей, %"); display(((hz / hz.loc["lgbm"] - 1) * 100).round(1))
m = pd.read_csv(f"{R}/metrics_val.csv")
C = m[m.subset == "общий подсет"]
display(C[C.slice.str.match("день:|событие|часы:")].pivot_table(index="slice", columns="model", values="wape").round(3))
for n in ["lgbm", "lgbm_all"]:
    p = f"{R}/fi_{n}.csv"
    if os.path.exists(p):
        print(f"Важность признаков {n} (топ-15):"); display(pd.read_csv(p).head(15).round(3))

## 4. Архив для репозитория

In [ ]:
import zipfile
with zipfile.ZipFile("/kaggle/working/results_csv.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{R}/*.csv"):
        z.write(p, os.path.basename(p))
print(sorted(os.path.basename(p) for p in glob.glob(f"{R}/*.csv")))